In [ ]:
# 全局设置
import datetime as dt

from IPython.display import Markdown

In [ ]:
# # 如无数据可使用下面的代码生成 Demo 数据
# from QuantStudio.Core.QSObject import Panel
# from QuantStudio.Factor.HDF5DB import HDF5DB
# TargetDir = "../data/HDF5"
# if not os.path.isdir(TargetDir): os.makedirs(TargetDir, exist_ok=True)

# HDB = HDF5DB(args={"MainDir": TargetDir})
# HDB.connect()

# np.random.seed(0)
# nDT, nID = 100, 20
# IDs = [str(i).zfill(6) + ".SZ" for i in range(1, nID + 1)]
# DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(nDT)]

# # stock_cn_day_bar
# Data = {
#     "Open" : pd.DataFrame(np.random.rand(nDT, nID), index=DTs, columns=IDs),
#     "High" : pd.DataFrame(np.random.rand(nDT, nID), index=DTs, columns=IDs),
#     "Low" : pd.DataFrame(np.random.rand(nDT, nID), index=DTs, columns=IDs),
#     "Close" : pd.DataFrame(np.random.rand(nDT, nID), index=DTs, columns=IDs),
#     "Volume" : pd.DataFrame(np.random.rand(nDT, nID) * 100, index=DTs, columns=IDs)
# }
# Data = Panel(Data)
# HDB.writeData(data=Data, table_name="stock_cn_day_bar", if_exists="update")

# # stock_cn_industry
# Data = {
#     "Industry" : pd.DataFrame(np.repeat(np.random.choice(["Fin", "TMT", "Ind"], size=(nDT, 1)), axis=1, repeats=nID), index=DTs, columns=IDs, dtype=pd.StringDtype(storage="python")),
# }
# Data = Panel(Data)
# HDB.writeData(data=Data, table_name="stock_cn_industry", if_exists="update")

# # 删除多余的表
# for iTableName in HDB.TableNames:
#     if iTableName not in ['stock_cn_day_bar', 'stock_cn_industry']:
#         HDB.deleteTable(iTableName)

# HDF5DB

继承自 WritableFactorDB, 初始化方法 `__init__`:
* args: dict, 因子库参数集
* config_file: None 或者 str, 因子库的配置文件地址, None 表示使用默认配置文件, 默认文件名为 "HDF5DBConfig.json", 默认路径为参见: [配置文件](../基本框架.ipynb#QuantStudio-对象)

![HDF5 因子库](../images/HDF5因子库.png)

主目录下的每个文件夹表示一张因子表, 每个文件夹下的扩展名为 hdf5 的文件存储了一个因子的数据。每个因子数据的 [HDF5 文件](https://www.hdfgroup.org/) 的 root group 下有三个 dataset:
* ID: 存储因子的 ID 序列数据, shape=(None,), dtype=String, 编码为 utf-8。
* DateTime: 存储因子的时点序列数据, shape=(None,), dtype=float, 时点转换成 timestamp 存储。
* Data: 存储因子数据, shape=(None, None), 行数等于 DateTime 的长度, 列数等于 ID 的长度。double 类型的因子数据存储为 float64 类型，string 类型的因子数据存储为 String 类型(编码为 utf-8)，object 类型的因子数据存储为 vlen_dtype(np.uint8) 类型。

因子数据 HDF5 文件 root group 的 attribute 里存储了因子的元信息。因子表的元信息存储在表目录下的 `_TableInfo.h5` 文件(没有该文件说明还未写入过元信息)中。

锁目录下的 "LockFile" 文件为库锁，修改因子库、因子表以及因子相关信息（比如创建、重命名、删除等操作）时需要获取库锁。锁目录下每个文件夹代表一张因子表，每个文件夹里的 "LockFile" 文件为表锁，读写因子数据时需要获取表锁。

In [2]:
# HDF5DB
from QuantStudio.Factor.HDF5DB import HDF5DB

HDB = HDF5DB(args={"MainDir": "../Data/HDF5"}).connect()
print("参数集说明")
display(Markdown(HDB.Args.info()))

参数集说明


* Name(名称): <class 'str'>, 默认值 HDF5DB
* MainDir(主目录): <class 'pathlib.Path'>, 无默认值, 存放数据的主目录
* LockDir(锁目录): typing.Optional[typing.Annotated[pathlib.Path, PathType(path_type='dir')]], 默认值 None, 存放锁文件的目录, 默认 None 表示和主目录相同
* FileOpenRetryNum(文件打开重试次数): typing.Union[int, float], 默认值 inf, 打开数据文件错误时的重试次数
* ProcessLock(进程锁): <class 'bool'>, 默认值 True, 是否添加进程锁用于防止多进程间读写冲突

## 因子表

In [ ]:
# 因子表列表
print("因子表 : ", HDB.TableNames)

因子表 :  ['stock_cn_day_bar', 'stock_cn_industry']


In [7]:
# 因子表参数集
FT = HDB.getTable("stock_cn_day_bar", args={"LookBack": 4, "OnlyStartLookBack": False, "OnlyLookBackNontarget": False, "OnlyLookBackDT": False, "TargetDT": None})
print("参数说明")
display(Markdown(FT.Args.info()))

参数说明


* Name(名称): <class 'str'>, 无默认值
* LookBack(回溯天数): typing.Union[int, float], 默认值 0, 缺失填充回溯的天数
* OnlyStartLookBack(只起始日回溯): <class 'bool'>, 默认值 False, 如果为 True, 表示只对提取数据的第一个时点进行缺失填充, 之后的时点不填充
* OnlyLookBackNontarget(只回溯非目标日): <class 'bool'>, 默认值 False, 如果为 True, 表示只用不在提取时点序列中的数据进行缺失填充
* OnlyLookBackDT(只回溯时点): <class 'bool'>, 默认值 False, 如果为 True, 表示所有 ID 统一沿着时点字段进行回溯填充, 不单独填充
* TargetDT(目标时点): typing.Optional[datetime.datetime], 默认值 None, 非 None 表示只取该时点的值返回

In [9]:
# 因子列表
print(f"因子表 '{FT.Name}' 中的因子 : ", FT.FactorNames)

因子表 'stock_cn_day_bar' 中的因子 :  ['Close', 'High', 'Low', 'Open', 'Volume']


In [ ]:
# 因子表读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ"]
Data = FT.readData(factor_names=["open", "close"], ids=IDs, dts=DTs)
print("因子表数据")
print(Data)
print("固定 ID 的切片数据: ", Data.iloc[:, :, 0], sep="\n")

因子表数据
<class 'QuantStudio.Tools.QSObjects.Panel'>
Dimensions: 2 (items) x 5 (major_axis) x 1 (minor_axis)
Items axis: open to close
Major_axis axis: 2025-01-01 00:00:00 to 2025-01-05 00:00:00
Minor_axis axis: 000001.SZ to 000001.SZ
固定 ID 的切片数据: 
                open     close
2025-01-01  0.548814  0.374794
2025-01-02  0.978618  0.571591
2025-01-03  0.359508  0.711557
2025-01-04  0.158970  0.966372
2025-01-05  0.317983  0.376607


: 